# Summary

In this notebook, we create (1) a Biguery table with IPEDS data and (2) a text document with a data dictionary and other content explaining the text.

In [1]:
import os, sys
import pandas as pd
import pandas_gbq as pbq

from google.cloud import bigquery

utils_path = "../../../../ccc-policy_assistant/interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../../../../ccc-policy_assistant/data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()


## Read file from Cloud storage

In [2]:
gcs_bucket_name = "ccc-crawl_data_xb"
gcs_directory = "crawl_data/ipeds/zipcsv_files/prep"

# Get a list of files in the GCP bucket
# gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
#                     gcs_bucket_name=gcs_bucket_name,
#                     path=gcs_directory)


In [3]:
# Get the dataframe with table descriptions from the GCP
df_desc = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                        gcs_bucket_name=gcs_bucket_name,
                                        gcs_directory=gcs_directory,
                                        file_name="descriptions_2025Apr29.csv",
                                        exact=False)
df_desc.head()


,Unnamed: 0,file_name,file_type,num_cols,cols,description
0,0,al2023.csv,csv,70,"UNITID,LEXP100K,LCOLELYN,XLPBOOKS,LPBOOKS,XLEB...",BigQuery table name: al2023. \nSource CSV data...
1,1,c2023dep.csv,csv,42,"UNITID,CIPCODE,PTOTAL,PTOTALDE,PTOTALDES,PASSO...",BigQuery table name: c2023dep. \nSource CSV da...
2,2,c2023_a.csv,csv,65,"UNITID,CIPCODE,MAJORNUM,AWLEVEL,XCTOTALT,CTOTA...",BigQuery table name: c2023_a. \nSource CSV dat...
3,3,c2023_b.csv,csv,82,"UNITID,XCSTOTLT,CSTOTLT,XCSTOTLM,CSTOTLM,XCSTO...",BigQuery table name: c2023_b. \nSource CSV dat...
4,4,c2023_c.csv,csv,37,"UNITID,AWLEVELC,XCSTOTLT,CSTOTLT,XCSTOTLM,CSTO...",BigQuery table name: c2023_c. \nSource CSV dat...


## Read the IPEDS tables from Google Cloud and save in dataframes

In [4]:
dfs = []
for idx in df_desc.index:

    # Read this data from GCP
    df = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                       gcs_bucket_name=gcs_bucket_name,
                                       gcs_directory=gcs_directory,
                                       file_name=df_desc.loc[idx, "file_name"],
                                       exact=False)

    # Drop this unneeded column
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)

    # Add this to list
    dfs.append(dict(table_name=df_desc.loc[idx, "file_name"],
                    df=df))


## Save the dataframes to BigQuery

In [2]:
# Create a bigquery client
client = bigquery.Client(project=os.environ["GOOGLE_CLOUD_PROJECT"])

# Create a dataset if one doesn't already exist
dataset_name = "ipeds"
gct.create_dataset_if_not_exists(project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                 dataset_name=dataset_name)

# Load tables
for df_dict in dfs:

    # Load table
     gct.load_pandas_to_bigquery(project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                 df=df_dict["df"],
                                 dataset_id="{}.{}".format(os.environ["GOOGLE_CLOUD_PROJECT"],
                                                           dataset_name),
                                 table_name=df_dict["table_name"].replace(".csv", ""),
                                 if_exists="replace"
                                 )




Dataset eternal-bongo-435614-b9.ipeds already exists


## List tables in BQ

In [5]:
tables = gct.list_bigquery_tables(project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                  dataset_name=dataset_name)
tables

print(len(tables))

41
